In [27]:
# 1. Cài đặt thư viện
!pip install ultralytics clearml -q


In [28]:
import yaml
import os
from ultralytics import YOLO


# THAY ĐỔI DÒNG NÀY: Sửa 'parking-smart-dataset' thành tên dataset bạn đã đặt lúc upload
DATASET_DIR = "/kaggle/input/datasets/nguynhongtngphong/parking-smart/DS_OK_Augmented" 

# Thư mục lưu kết quả (Read-Write)
WORKING_DIR = "/kaggle/working"


In [29]:
# 2. Tạo file data.yaml động (Lưu vào thư mục working)
yaml_content = {
    'train': os.path.join(DATASET_DIR, 'images', 'train'),
    'val': os.path.join(DATASET_DIR, 'images', 'val'),
    'test': os.path.join(DATASET_DIR, 'images', 'test'),
    'nc': 1,
    'names': ['car']
}

yaml_path = os.path.join(WORKING_DIR, 'dataset_kaggle.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

In [30]:
# model = YOLO('yolov8n.pt') 

# results = model.train(
#     data=yaml_path,
#     epochs=50,
#     imgsz=640,
#     # TĂNG BATCH SIZE: Vì có 2 GPU, bạn có thể tăng batch size lên 32. 
#     # Mỗi GPU sẽ gánh 16 ảnh. Nếu báo lỗi hết VRAM (CUDA Out of memory), hãy hạ về 16.
#     batch=8,                  
#     device=[0, 1],                 # ĐIỂM QUYẾT ĐỊNH: Lệnh gọi cả 2 GPU số 0 và số 1 cùng chạy
#     project=WORKING_DIR,           
#     name='Model_Parking_Smart_2GPU',    
#     patience=15,
#     save=True
# )

In [31]:
# model = YOLO('yolov8n.pt') 

# results = model.train(
#     data=yaml_path,
#     epochs=50,
#     imgsz=640,
#     batch=8,                      # T4x2 của Kaggle dư sức gánh batch 32
#     device=[0, 1],                 # Kích hoạt cả 2 GPU
#     project=WORKING_DIR,           
#     name='Model_Parking_V2',       
#     degrees=90.0,                  # Xoay xe lộn xộn mọi hướng (Vì nhìn từ trên xuống xe đỗ hướng nào cũng giống nhau)
#     perspective=0.001,             # Bẻ cong méo mó phối cảnh 3D
#     shear=10.0,                    # Kéo xiên, bẹp hình dáng xe
#     scale=0.5,                     # Phóng to/thu nhỏ (Mô phỏng việc Camera lắp quá cao hoặc quá thấp)
#     patience=15,
#     save=True
# )

In [32]:
# Tải model đã huấn luyện xong ở trên
OLD_WEIGHTS = '/kaggle/input/datasets/nguynhongtngphong/model-v7/best-v7.pt' 
model = YOLO(OLD_WEIGHTS)

# Đường dẫn đến bộ dữ dữ liệu quay từ trên cao bằng flycam
FLYCAM_DIR = '/kaggle/input/datasets/nguynhongtngphong/data-oke/data_oke/' 

WORKING_DIR = "/kaggle/working"

In [33]:
# Tạo file data.yaml động (Lưu vào thư mục working)
yaml_content = {
    'train': os.path.join(FLYCAM_DIR, 'images', 'train'),
    'val': os.path.join(FLYCAM_DIR, 'images', 'val'),
    'nc': 1,
    'names': ['car']
}

yaml_path = os.path.join(WORKING_DIR, 'dataset_kaggle_2.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

In [34]:

results = model.train(
    data=yaml_path,
    epochs=30,               
    imgsz=640,
    batch=32,                
    device=[0, 1],           
    lr0=0.001,               
    warmup_epochs=2,         
    degrees=90.0,
    project='/kaggle/working/',           
    name='Model_Parking_All-V2',
    save=True
)

print("Hoàn thành!")

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_kaggle_2.yaml, degrees=90.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/nguynhongtngphong/model-v7/best-v7.pt, momentum=0.937, mosaic=1